# 03 — The k dial

*Module 01 · k-Nearest Neighbours — notebook 3 of 6*

This is the third and last of the fundamentals notebooks, and it puts **k** itself under the lens. In
notebook 1, k was the number of neighbours that vote; we set it to 5 and moved on. But k is a choice
with real consequences — turn it one way and k-NN memorizes every point, turn it the other and it
smooths the data into mush. k is a **dial**, and this notebook is about reading it and setting it
honestly.

The good news: you already have the tools. Over- and under-fitting (notebook 09) and cross-validation
(notebook 10) were built for exactly this moment.

**Prerequisites:** notebooks 01–02 (the vote, distance, scale), and from module 00: notebook 05 (the
decision boundary), notebook 09 (over- and under-fitting, the generalization gap), notebook 10
(cross-validation).

**What you will be able to do**

- See k as the **bias–variance knob**: small k flexible and local, large k smooth and global.
- Recognize k = 1 as memorizing (overfit) and very large k as over-smoothing (underfit).
- Diagnose the dial with train and test error.
- **Choose k by cross-validation on the training data**, then evaluate the choice once on the test set.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import StratifiedKFold, train_test_split

from ml_course import viz
from ml_course.colors import COLORS

# Fix the seed so every run tells the same story.
np.random.seed(0)
viz.use_course_style()

X, y = make_moons(n_samples=300, noise=0.30, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=0
)


# The by-hand k-NN vote (notebook 1).
def knn_predict(X_ref, y_ref, queries, k=5):
    X_ref = np.asarray(X_ref, dtype=float)
    queries = np.atleast_2d(np.asarray(queries, dtype=float))
    predictions = []
    for q in queries:
        dist = np.sqrt(((X_ref - q) ** 2).sum(axis=1))
        nearest = np.argsort(dist)[:k]
        predictions.append(np.bincount(y_ref[nearest], minlength=2).argmax())
    return np.array(predictions)


# A tiny fit/predict wrapper (notebook 2) so we can reuse the boundary plotter.
class ByHandKNN:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X_ref, y_ref):
        self.X_ref = np.asarray(X_ref, dtype=float)
        self.y_ref = np.asarray(y_ref)
        return self

    def predict(self, queries):
        return knn_predict(self.X_ref, self.y_ref, queries, self.k)


print(f"training points: {X_train.shape[0]}   test points: {X_test.shape[0]}")

## A recap, and the question this notebook asks

Four ideas meet here:

- **The vote and k** (notebook 01) — k is the number of neighbours that vote.
- **Distance** (notebook 02) — how "nearest" is measured. We use the original moons, whose two
  features are already on comparable scales (notebook 02 taught us to standardize when they are not).
- **Over- and under-fitting** (notebook 09) — a model can fit its training data too well and fail to
  generalize; training error alone hides this.
- **Cross-validation** (notebook 10) — how to estimate out-of-sample performance, and choose between
  models, using only the training data.

The question: **what value of k?** Let us turn the dial and watch.

## k is a dial

Picture the extremes. With **k = 1**, a new point copies its single nearest neighbour — the boundary
bends around every individual point, including noisy ones. With a **very large k**, a point is
labelled by a vote among a huge crowd of neighbours, most of them far away — the boundary becomes
smooth and barely responds to local structure. Everything useful lives between these extremes. We
will first *see* this in the decision boundary, then *measure* it.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, k in zip(axes, (1, 15, 151), strict=True):
    model = ByHandKNN(k=k).fit(X_train, y_train)
    viz.plot_decision_boundary(model, X_train, y_train, resolution=200, ax=ax)
    ax.set_title(f"k = {k}")
    ax.set_xlabel("feature 1")
    ax.set_ylabel("feature 2")
fig.tight_layout()
plt.show()

**Read the figure.** Three settings of the same dial (the dots are the **training** points). At
**k = 1** the boundary is jagged and carves out little islands around single points — k-NN is
memorizing the training set, noise and all; notice that every training point sits on its own side,
the perfect-recall mirage the next section takes apart. At **k = 15** the boundary is smooth and
traces the two crescents cleanly. At **k = 151** it has gone too far the other way: the vote is
dominated by distant points, and the boundary is a broad sweep that ignores the real shape. Compare this with notebook 05's nearest-centroid classifier, whose boundary
was a single straight line — one global rule. k-NN's boundary is **local**, and k sets how local it is
allowed to be. That is k-NN's inductive bias.

## Does k = 1's perfect memory make it the best?

At k = 1, every training point's nearest neighbour is itself, so k-NN labels the training set with
**zero error** — flawless recall. It is tempting to read that as "k = 1 is the best model." Notebook
09 taught us why that is a trap: training accuracy measures memorization, not generalization. The real
question is how the dial behaves on **unseen** data. Let us turn it across a range of k and measure
both train and test error. (We sweep *odd* values of k so a two-class vote can never tie 2–2; what to
do when k is even is a question we open in notebook 4.)

In [ ]:
ks = [1, 3, 5, 9, 15, 25, 51, 75, 101, 151, 201]
train_err = [1 - (knn_predict(X_train, y_train, X_train, k) == y_train).mean() for k in ks]
test_err = [1 - (knn_predict(X_train, y_train, X_test, k) == y_test).mean() for k in ks]

fig = viz.plot_train_test_curve(
    ks, train_err, test_err, xlabel="k (neighbours, log scale)", ylabel="error"
)
fig.axes[0].set_xscale("log")
plt.show()

**Read the figure.** Two curves as k grows (note the log scale on k). **Training error** starts at
exactly 0 for k = 1 — perfect memorization — and rises as larger k forces the model to generalize over
more neighbours. **Test error** tells the real story: it is a shallow U. It is higher at k = 1 (the
model overfit the training noise), dips to its best in the middle, then climbs steeply for large k as
the model underfits. The gap between the two curves at small k — training error far below test error —
points to overfitting (notebook 09 was careful that this gap *indicates* overfitting rather than being
the variance itself).

## Choosing k honestly

There is a catch in what we did: we read the **test** error to see the U. That is acceptable for a
lesson, but in a real project the test set must stay sealed until the very end — if you choose k by
watching the test error, you are quietly fitting to the test set, and your final score will be
optimistic (notebook 10's cardinal sin). So we choose k the honest way: **cross-validation on the
training data only.** We split the training set into folds, and for each k we average the validation
accuracy across folds. The test set is not touched.

In [ ]:
# Cross-validation on the TRAINING set only (notebook 10), entirely by hand.
def cv_accuracy(features, labels, k, n_splits=5):
    folds = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)
    scores = []
    for train_idx, val_idx in folds.split(features, labels):
        pred = knn_predict(features[train_idx], labels[train_idx], features[val_idx], k)
        scores.append((pred == labels[val_idx]).mean())
    return np.mean(scores)


cv_acc = [cv_accuracy(X_train, y_train, k) for k in ks]
best_k = ks[int(np.argmax(cv_acc))]
print(f"CV-chosen k = {best_k}   (cross-validation accuracy {max(cv_acc):.3f})")

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(ks, cv_acc, color=COLORS["model"], marker="o", linewidth=2)
ax.axvline(best_k, color=COLORS["highlight"], linestyle="--", label=f"CV-best k = {best_k}")
ax.set_xscale("log")
ax.set_xlabel("k (neighbours, log scale)")
ax.set_ylabel("cross-validation accuracy")
ax.legend()
plt.show()

**Read the figure.** This curve is built **without ever touching the test set** — it is five-fold
cross-validation on the training data alone (notebook 10). Cross-validation accuracy rises from k = 1,
peaks at **k = 15**, then falls as large k underfits. So our principled choice is k = 15. (You may
notice the test-error curve earlier bottomed out at k = 3, not 15. That is expected: cross-validation
is an estimate, and a single test split is another — they will not pinpoint the same k; the CV folds
even carry their own random seed, so this curve too would wobble a little under a different one. The
discipline is to choose on the training data and not go back to pick whatever happened to score best
on test.)

In [ ]:
# Evaluate the CV-chosen k once on the sealed test set, with the extremes for contrast.
for k in (1, best_k, 151):
    acc = (knn_predict(X_train, y_train, X_test, k) == y_test).mean()
    tag = "   <- chosen by CV" if k == best_k else ""
    print(f"test accuracy at k = {k:3d}: {acc:.3f}{tag}")

**Read the output.** The k chosen by cross-validation, k = 15, scores **0.956** on the test set it
never saw — comfortably above the overfit k = 1 (0.933) and far above the underfit k = 151 (0.678).
This is the honest workflow in one line: **cross-validation chooses k on the training data; the test
set confirms the choice exactly once.**

## The bias–variance trade-off

The dial has a name you met in notebook 09. A **small k** makes few assumptions and bends to fit the
data closely: **low bias**, but **high variance** — it sways with every noisy point (overfitting). A
**large k** averages over many neighbours: **low variance**, steady and smooth, but **high bias** — it
cannot follow the real structure (underfitting). k is the knob between the two, and cross-validation is
how we find the setting that trades them off best. (We are reading variance here off the jagged
boundary and the train–test gap on one split; notebook 09 measured it directly by refitting on many
resampled training sets — small k is the flexible, high-variance end, like the high-degree polynomial
there.) Every method in this course has a version of this trade-off; for k-NN, it is k.

## Your turn

Reason these through.

1. At k = 1, k-NN has zero error on the training set. Why is that *not* evidence that k = 1 is the
   best choice?
2. We chose k by cross-validation on the training data rather than by the test score. Why does
   choosing on the test score give an over-optimistic picture of how the model will do in the wild?
3. A colleague says: "I tried every k from 1 to 50 and k = 3 gave the highest accuracy *on the test
   set*, so I'll ship k = 3." What is the methodological problem, and what should they have done?

## What you built

You turned k from a number we picked into a dial you can read. You saw its two failure modes — k = 1
memorizing the training set (overfit), large k smoothing the data away (underfit) — in the decision
boundary and in the train/test error curves. Then you chose k the honest way: cross-validation on the
training data picked k = 15, and the sealed test set confirmed it at 0.956 (these exact values are particular to
this dataset and seed; the *pattern* — overfit at small k, underfit at large k, cross-validation
finding the middle — is the lesson). With that, you have closed the k-NN fundamentals: the vote, the
distance and its scale, and the dial that sets the trade-off.

**Vocabulary**

- **the k dial** — k as the control between a flexible, local model and a smooth, global one.
- **bias–variance trade-off** — small k: low bias, high variance; large k: high bias, low variance.
- **overfitting / underfitting** — fitting noise (small k) versus missing real structure (large k).
- **cross-validation for model selection** — choosing k by validation accuracy on the training folds,
  never the test set.
- **hyperparameter** — a setting like k that you choose, rather than one the model fits from the data.

## References

- G. James, D. Witten, T. Hastie, R. Tibshirani (2021). *An Introduction to Statistical Learning* —
  §2.2.3 (k-NN), §2.2.2 (the bias–variance trade-off), §5.1 (cross-validation). DOI:
  [10.1007/978-1-0716-1418-1](https://doi.org/10.1007/978-1-0716-1418-1).
- T. Hastie, R. Tibshirani, J. Friedman (2009). *The Elements of Statistical Learning* — §2.5 (k-NN),
  §7.10 (cross-validation and model selection). DOI:
  [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7).

---

*Previous: 02 — Distance & the scale trap* · *Next: 04 — The estimator & its parameters*